In [1]:
import os
import json
import joblib
import pandas as pd
import numpy as np
try:
    import xgboost as xgb
except:
    !pip install "xgboost<3"
    import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss, accuracy_score

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


#### Constants

In [2]:
str_bucket = os.getcwd().split('/')[4].replace('_','-')
print(f'Bucket: {str_bucket}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

str_dirname_output = './output'
os.makedirs(str_dirname_output, exist_ok=True)

Bucket: assessment-swish-analytics
Task: 04_model


#### Import Data from S3

In [3]:
X_train = pd.read_csv(f's3://{str_bucket}/03_preprocessing/X_train.csv')
X_valid = pd.read_csv(f's3://{str_bucket}/03_preprocessing/X_valid.csv')
X_test = pd.read_csv(f's3://{str_bucket}/03_preprocessing/X_test.csv')

y_train = pd.read_csv(f's3://{str_bucket}/03_preprocessing/y_train.csv').squeeze()
y_valid = pd.read_csv(f's3://{str_bucket}/03_preprocessing/y_valid.csv').squeeze()
y_test = pd.read_csv(f's3://{str_bucket}/03_preprocessing/y_test.csv').squeeze()

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_valid: {X_valid.shape}, y_valid: {y_valid.shape}')
print(f'X_test:  {X_test.shape}, y_test:  {y_test.shape}')

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/fsspec/registry.py:301: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


X_train: (538294, 89), y_train: (538294,)
X_valid: (120479, 89), y_valid: (120479,)
X_test:  (39545, 89), y_test:  (39545,)


#### Drop Target-Derived Features

The mix features (`pitcher_pct_*`, `pitcher_count_pct_*`, `pitcher_hand_pct_*`, `pitcher_hand_count_pct_*`, `batter_pct_*`) are computed from training set targets. Even with Bayesian smoothing, these features allow the model to memorize training label distributions, causing a large train-valid accuracy gap. Removing them forces the model to learn from game-situation and count-based features that generalize better.

In [4]:
# identify target-derived mix features
list_mix_prefixes = ['pitcher_pct_', 'pitcher_count_pct_', 'pitcher_hand_pct_', 'pitcher_hand_count_pct_', 'batter_pct_']
list_mix_cols = [c for c in X_train.columns if any(c.startswith(p) for p in list_mix_prefixes)]

print(f'Dropping {len(list_mix_cols)} target-derived mix features:')
for col in sorted(list_mix_cols):
    print(f'  {col}')

X_train = X_train.drop(columns=list_mix_cols)
X_valid = X_valid.drop(columns=list_mix_cols)
X_test = X_test.drop(columns=list_mix_cols)

print(f'\nRemaining features: {X_train.shape[1]}')

Dropping 40 target-derived mix features:
  batter_pct_CH
  batter_pct_CU
  batter_pct_FC
  batter_pct_FF
  batter_pct_FS
  batter_pct_FT
  batter_pct_SI
  batter_pct_SL
  pitcher_count_pct_CH
  pitcher_count_pct_CU
  pitcher_count_pct_FC
  pitcher_count_pct_FF
  pitcher_count_pct_FS
  pitcher_count_pct_FT
  pitcher_count_pct_SI
  pitcher_count_pct_SL
  pitcher_hand_count_pct_CH
  pitcher_hand_count_pct_CU
  pitcher_hand_count_pct_FC
  pitcher_hand_count_pct_FF
  pitcher_hand_count_pct_FS
  pitcher_hand_count_pct_FT
  pitcher_hand_count_pct_SI
  pitcher_hand_count_pct_SL
  pitcher_hand_pct_CH
  pitcher_hand_pct_CU
  pitcher_hand_pct_FC
  pitcher_hand_pct_FF
  pitcher_hand_pct_FS
  pitcher_hand_pct_FT
  pitcher_hand_pct_SI
  pitcher_hand_pct_SL
  pitcher_pct_CH
  pitcher_pct_CU
  pitcher_pct_FC
  pitcher_pct_FF
  pitcher_pct_FS
  pitcher_pct_FT
  pitcher_pct_SI
  pitcher_pct_SL

Remaining features: 49


#### Encode Target Labels

In [5]:
le = LabelEncoder()
le.fit(y_train)

y_train_enc = le.transform(y_train)
y_valid_enc = le.transform(y_valid)
y_test_enc = le.transform(y_test)

print(f'Classes: {le.classes_}')
print(f'Number of classes: {len(le.classes_)}')

Classes: ['CH' 'CU' 'FC' 'FF' 'FS' 'FT' 'SI' 'SL']
Number of classes: 8


#### Create DMatrices

In [6]:
dtrain = xgb.DMatrix(X_train, label=y_train_enc)
dvalid = xgb.DMatrix(X_valid, label=y_valid_enc)
dtest = xgb.DMatrix(X_test, label=y_test_enc)

#### Model Parameters

In [ ]:
dict_params = {
    'objective': 'multi:softprob',
    'num_class': len(le.classes_),
    'eval_metric': 'mlogloss',
    'learning_rate': 0.03,
    'max_depth': 3,
    'min_child_weight': 50,
    'subsample': 0.7,
    'colsample_bytree': 0.5,
    'gamma': 0.5,
    'reg_lambda': 5,
    'seed': 42,
    'verbosity': 0,
}

print('Model parameters:')
for k, v in dict_params.items():
    print(f'  {k}: {v}')

#### Train Model

In [8]:
list_features = list(X_train.columns)
print(f'Training with {len(list_features)} features')

# build DMatrices
dtrain = xgb.DMatrix(X_train, label=y_train_enc)
dvalid = xgb.DMatrix(X_valid, label=y_valid_enc)
dtest = xgb.DMatrix(X_test, label=y_test_enc)

# train model
dict_params['verbosity'] = 1
model = xgb.train(
    dict_params,
    dtrain,
    num_boost_round=1000,
    evals=[(dtrain, 'train'), (dvalid, 'valid')],
    early_stopping_rounds=100,
    verbose_eval=100,
)
dict_params['verbosity'] = 0

print(f'\nBest iteration: {model.best_iteration}')
print(f'Best validation mlogloss: {model.best_score:.4f}')

Training with 49 features
[0]	train-mlogloss:2.05418	valid-mlogloss:2.06379
[100]	train-mlogloss:1.56249	valid-mlogloss:2.01727
[131]	train-mlogloss:1.54903	valid-mlogloss:2.06012

Best iteration: 32
Best validation mlogloss: 1.9229


#### Overfitting Assessment: Train vs. Validation vs. Test

Compare accuracy and log-loss across all three splits to assess generalization. A well-fit model should have similar metrics across splits; large gaps indicate overfitting.

In [9]:
# predictions on all three splits using best iteration
int_best_iter = model.best_iteration + 1

arr_train_proba = model.predict(dtrain, iteration_range=(0, int_best_iter))
arr_valid_proba = model.predict(dvalid, iteration_range=(0, int_best_iter))
arr_test_proba = model.predict(dtest, iteration_range=(0, int_best_iter))

arr_train_pred = le.classes_[arr_train_proba.argmax(axis=1)]
arr_valid_pred = le.classes_[arr_valid_proba.argmax(axis=1)]
arr_test_pred = le.classes_[arr_test_proba.argmax(axis=1)]

# compute metrics
dict_overfit = {
    'Split': ['Train', 'Validation', 'Test'],
    'Accuracy': [
        accuracy_score(y_train, arr_train_pred),
        accuracy_score(y_valid, arr_valid_pred),
        accuracy_score(y_test, arr_test_pred),
    ],
    'Log-Loss': [
        log_loss(y_train_enc, arr_train_proba),
        log_loss(y_valid_enc, arr_valid_proba),
        log_loss(y_test_enc, arr_test_proba),
    ],
    'N Samples': [len(y_train), len(y_valid), len(y_test)],
}
df_overfit = pd.DataFrame(dict_overfit)

# compute gaps
flt_train_acc = dict_overfit['Accuracy'][0]
flt_valid_acc = dict_overfit['Accuracy'][1]
flt_test_acc = dict_overfit['Accuracy'][2]
flt_train_loss = dict_overfit['Log-Loss'][0]
flt_valid_loss = dict_overfit['Log-Loss'][1]
flt_test_loss = dict_overfit['Log-Loss'][2]

print('=' * 60)
print('OVERFITTING ASSESSMENT')
print('=' * 60)
print(df_overfit.to_string(index=False))
print('-' * 60)
print(f'Train-Valid accuracy gap:  {flt_train_acc - flt_valid_acc:+.4f}')
print(f'Valid-Test accuracy gap:   {flt_valid_acc - flt_test_acc:+.4f}')
print(f'Train-Valid log-loss gap:  {flt_valid_loss - flt_train_loss:+.4f}')
print(f'Valid-Test log-loss gap:   {flt_test_loss - flt_valid_loss:+.4f}')
print('=' * 60)

if (flt_train_acc - flt_valid_acc) > 0.10:
    print('\nWARNING: Train-Valid accuracy gap > 10% -- model is overfitting.')
    print('  Consider: stronger regularization, fewer features, or more data.')
elif (flt_train_acc - flt_valid_acc) > 0.05:
    print('\nCAUTION: Train-Valid accuracy gap > 5% -- mild overfitting detected.')
else:
    print('\nTrain-Valid gap is small -- model generalizes reasonably well.')

OVERFITTING ASSESSMENT
     Split  Accuracy  Log-Loss  N Samples
     Train  0.415570  1.682364     538294
Validation  0.296010  1.922916     120479
      Test  0.371905  1.763144      39545
------------------------------------------------------------
Train-Valid accuracy gap:  +0.1196
Valid-Test accuracy gap:   -0.0759
Train-Valid log-loss gap:  +0.2406
Valid-Test log-loss gap:   -0.1598

  Consider: stronger regularization, fewer features, or more data.


#### Save Model and Artifacts Locally

In [10]:
# save model locally
str_filename = 'xgb_model.json'
str_local_path = f'{str_dirname_output}/{str_filename}'
model.save_model(str_local_path)
print(f'Saved model to {str_local_path}')

# save label encoder locally
str_filename = 'label_encoder.joblib'
str_local_path = f'{str_dirname_output}/{str_filename}'
joblib.dump(le, str_local_path)
print(f'Saved label encoder to {str_local_path}')

# save best iteration for downstream use
str_filename = 'best_iteration.json'
str_local_path = f'{str_dirname_output}/{str_filename}'
with open(str_local_path, 'w') as f:
    json.dump({'best_iteration': model.best_iteration}, f)
print(f'Saved best_iteration ({model.best_iteration}) to {str_local_path}')

# save overfitting metrics
str_filename = 'overfitting_metrics.json'
str_local_path = f'{str_dirname_output}/{str_filename}'
with open(str_local_path, 'w') as f:
    json.dump({
        'train_accuracy': flt_train_acc,
        'valid_accuracy': flt_valid_acc,
        'test_accuracy': flt_test_acc,
        'train_logloss': flt_train_loss,
        'valid_logloss': flt_valid_loss,
        'test_logloss': flt_test_loss,
        'train_valid_acc_gap': flt_train_acc - flt_valid_acc,
        'valid_test_acc_gap': flt_valid_acc - flt_test_acc,
    }, f, indent=2)
print(f'Saved overfitting metrics to {str_local_path}')

# save test predictions locally
arr_test_proba = model.predict(dtest, iteration_range=(0, model.best_iteration + 1))
arr_test_pred = le.classes_[arr_test_proba.argmax(axis=1)]

df_test_preds = pd.DataFrame(arr_test_proba, columns=[f'prob_{c}' for c in le.classes_])
df_test_preds['predicted'] = arr_test_pred
df_test_preds['actual'] = y_test.values
str_filename = 'test_predictions.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df_test_preds.to_csv(str_local_path, index=False)
print(f'Saved test predictions to {str_local_path}')

# save training params locally
dict_params_save = dict_params.copy()
dict_params_save['best_iteration'] = model.best_iteration
dict_params_save['best_score'] = model.best_score
dict_params_save['n_features'] = len(list_features)
str_filename = 'params.json'
str_local_path = f'{str_dirname_output}/{str_filename}'
with open(str_local_path, 'w') as f:
    json.dump(dict_params_save, f, indent=2)
print(f'Saved params to {str_local_path}')

Saved model to ./output/xgb_model.json
Saved label encoder to ./output/label_encoder.joblib
Saved best_iteration (32) to ./output/best_iteration.json
Saved overfitting metrics to ./output/overfitting_metrics.json
Saved test predictions to ./output/test_predictions.csv
Saved params to ./output/params.json
